# Анализ данных Excel-отчёта бизнеса (январь 2026)

Цель: только разбор содержимого xlsx, без обращения к Impala/lake.

**Что узнаём**:
1. Сколько строк / колонок в файле,
2. Полный список колонок и их типы,
3. Сколько уникальных ИНН и `agr_id` (если есть в Excel),
4. Дубли по `ИНН` и `agr_id`,
5. Распределение длин ИНН (10 для ЮЛ / 12 для ИП),
6. Заполненность каждой колонки (null / zero / non-zero),
7. Распределение KPI (`ЧОД`, `Фин.Рез`, `ТТ`, `Терминалы`, `АУР`, `Амортизация`, комиссии),
8. Сэмплы строк разных типов (всё нули / только АУР / с оборотом).

In [ ]:
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 0) Параметры

Меняй только `excel_path` и `excel_header` если нужно.

In [ ]:
excel_path = '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx'
excel_header = 1

# Алиасы — для гибкого поиска колонок
inn_aliases             = ['ИНН', 'INN']
agr_id_aliases          = ['agr_id', 'AGR_ID', 'ID договора', 'ИД договора', 'ID Договора эквайринга', 'ID договора эквайринга']
contract_num_aliases    = ['Номер договора', 'c_agr_number', 'Номер договора эквайринга']
chod_aliases            = ['ЧОД', 'CHOD']
finrez_aliases          = ['Фин.Рез', 'Фин. Рез.', 'ФИН.РЕЗ', 'ФИНРЕЗ']
term_cnt_aliases        = ['Количество терминалов', 'Кол-во терминалов', 'Терминалы', 'term_cnt']
retl_cnt_aliases        = ['Количество торговых точек', 'Кол-во торговых точек', 'ТТ', 'retl_cnt', 'Торговые точки']
trx_sum_aliases         = ['Сумма операций', 'Оборот', 'sum_trx_amt', 'trx_sum']
trx_cnt_aliases         = ['Количество операций', 'Кол-во операций', 'cnt_trx', 'trx_cnt']
aur_aliases             = ['АУР', 'AUR']
amort_aliases           = ['Амортизация', 'amortization']
comm_total_aliases      = ['Комиссия эквайринга', 'commission_total']
comm_from_ops_aliases   = ['Комиссия \n(% с операций)', 'Комиссия (% с операций)', 'commission_from_ops']
comm_monthly_aliases    = ['Комиссия \n(₽ в месяц)', 'Комиссия (₽ в месяц)', 'commission_monthly', 'Фикс комиссия']
int_component_aliases   = ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, руб)', 'int_component', 'IRF']

print(f'Файл: {excel_path}')
print(f'Header: {excel_header}')

## 1) Загрузка файла и общая структура

In [ ]:
def normalize_inn(value):
    if pd.isna(value):
        return None
    s = re.sub(r'\D+', '', str(value))
    return s if s else None


def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    norm_map = {normalize_colname(c): c for c in columns}
    for alias in aliases:
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    return None


def parse_numeric(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace('\xa0', '').replace(' ', '')
    if not s:
        return np.nan
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    s = re.sub(r'[^0-9\.-]', '', s)
    if s in {'', '-', '.', '-.'}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


excel_df = pd.read_excel(excel_path, header=excel_header)

print(f'Всего строк: {len(excel_df):,}')
print(f'Всего колонок: {len(excel_df.columns)}')
print(f'\nРазмер DataFrame в памяти: {excel_df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB')

## 2) Полный список колонок с типами

In [ ]:
columns_info = pd.DataFrame({
    'column_name': excel_df.columns,
    'dtype': [str(excel_df[c].dtype) for c in excel_df.columns],
    'non_null_cnt': [int(excel_df[c].notna().sum()) for c in excel_df.columns],
    'null_cnt': [int(excel_df[c].isna().sum()) for c in excel_df.columns],
    'unique_cnt': [int(excel_df[c].nunique(dropna=True)) for c in excel_df.columns],
})

columns_info['fill_rate_pct'] = (columns_info['non_null_cnt'] / max(len(excel_df), 1) * 100).round(2)

print('Полный список колонок:')
display(columns_info)

## 3) Поиск ключевых колонок (ИНН, agr_id, KPI)

Идём по списку алиасов и фиксируем, какие из ожидаемых колонок есть в файле, а какие отсутствуют.

In [ ]:
column_search = {
    'inn':              inn_aliases,
    'agr_id':           agr_id_aliases,
    'contract_number':  contract_num_aliases,
    'chod':             chod_aliases,
    'finrez':           finrez_aliases,
    'term_cnt':         term_cnt_aliases,
    'retl_cnt':         retl_cnt_aliases,
    'trx_sum':          trx_sum_aliases,
    'trx_cnt':          trx_cnt_aliases,
    'aur':              aur_aliases,
    'amortization':     amort_aliases,
    'commission_total':       comm_total_aliases,
    'commission_from_ops':    comm_from_ops_aliases,
    'commission_monthly':     comm_monthly_aliases,
    'int_component':          int_component_aliases,
}

resolved = {}
for label, aliases in column_search.items():
    found = pick_column(excel_df.columns, aliases)
    resolved[label] = found

resolved_df = pd.DataFrame([{'logical': k, 'excel_column': v if v is not None else '[НЕ НАЙДЕНО]'} for k, v in resolved.items()])
print('Сопоставление логических полей и колонок Excel:')
display(resolved_df)

## 4) Уникальные ИНН и `agr_id`

Главные показатели объёма данных:
- сколько строк в Excel,
- сколько уникальных ИНН,
- сколько уникальных `agr_id` (если колонка есть),
- дубли по ИНН и `agr_id`.

In [ ]:
if resolved['inn'] is not None:
    excel_df['inn_norm'] = excel_df[resolved['inn']].apply(normalize_inn)
else:
    excel_df['inn_norm'] = None

if resolved['agr_id'] is not None:
    excel_df['agr_id_norm'] = excel_df[resolved['agr_id']].astype(str).str.strip()
    excel_df.loc[excel_df['agr_id_norm'].isin(['nan', 'None', '']), 'agr_id_norm'] = None
else:
    excel_df['agr_id_norm'] = None

n_rows = len(excel_df)
n_inn  = excel_df['inn_norm'].dropna().nunique()
n_agr  = excel_df['agr_id_norm'].dropna().nunique() if resolved['agr_id'] else 0

print(f'Всего строк: {n_rows:,}')
print(f'Уникальных ИНН: {n_inn:,}')
if resolved['agr_id']:
    print(f'Уникальных agr_id: {n_agr:,}')
else:
    print('Колонка agr_id не найдена в Excel')

# Дубли
dup_inn = excel_df.groupby('inn_norm').size()
dup_inn = dup_inn[dup_inn > 1]
print(f'\nИНН с >1 строкой: {len(dup_inn):,} (всего таких строк: {int(dup_inn.sum()):,})')

if resolved['agr_id']:
    dup_agr = excel_df.groupby('agr_id_norm').size()
    dup_agr = dup_agr[dup_agr > 1]
    print(f'agr_id с >1 строкой: {len(dup_agr):,} (всего таких строк: {int(dup_agr.sum()):,})')
else:
    dup_agr = pd.Series(dtype=int)

# Распределение длин ИНН
print('\nРаспределение длин ИНН (10 = ЮЛ, 12 = ИП):')
print(excel_df['inn_norm'].dropna().str.len().value_counts().sort_index())

# Если есть и ИНН, и agr_id — проверим, сколько agr_id приходится на 1 ИНН
if resolved['agr_id']:
    agr_per_inn = excel_df.groupby('inn_norm')['agr_id_norm'].nunique()
    print('\nРаспределение количества agr_id на 1 ИНН:')
    print(agr_per_inn.value_counts().sort_index().head(15))

## 5) Заполненность всех KPI

Для каждого KPI считаем: `null`, `zero`, `negative`, `positive`. Это покажет, какие комбинации действительно есть в файле.

In [ ]:
numeric_cols_logical = ['chod', 'finrez', 'term_cnt', 'retl_cnt', 'trx_sum', 'trx_cnt',
                        'aur', 'amortization', 'commission_total', 'commission_from_ops',
                        'commission_monthly', 'int_component']

numeric_in_excel = {}
for label in numeric_cols_logical:
    src = resolved.get(label)
    if src is None:
        continue
    excel_df[f'{label}_num'] = excel_df[src].apply(parse_numeric)
    numeric_in_excel[label] = f'{label}_num'

print('Распределение значений по KPI:\n')
fill_rows = []
for label, num_col in numeric_in_excel.items():
    s = excel_df[num_col]
    fill_rows.append({
        'logical_field': label,
        'excel_column': resolved[label],
        'null':         int(s.isna().sum()),
        'zero':         int((s == 0).sum()),
        'positive':     int((s > 0).sum()),
        'negative':     int((s < 0).sum()),
        'min':          float(s.min()) if s.notna().any() else None,
        'max':          float(s.max()) if s.notna().any() else None,
        'sum':          float(s.sum(skipna=True)),
        'mean':         float(s.mean(skipna=True)) if s.notna().any() else None,
    })

fill_df = pd.DataFrame(fill_rows)
display(fill_df)

## 6) Кресты по статусам активности

Главная задача — понять, какие комбинации `chod=0` / `term_cnt=0` / `retl_cnt=0` / `aur=0` встречаются в Excel.

Это покажет, **по какому признаку** бизнес отбирает строки в свой отчёт.

In [ ]:
def has_value(col):
    if col not in excel_df.columns:
        return None
    return excel_df[col].fillna(0) > 0

flags = {}
for label in ['chod', 'finrez', 'term_cnt', 'retl_cnt', 'trx_sum', 'trx_cnt', 'aur', 'amortization']:
    num_col = f'{label}_num'
    if num_col in excel_df.columns:
        flags[f'has_{label}'] = excel_df[num_col].fillna(0) > 0

flags_df = pd.DataFrame(flags, index=excel_df.index)
print('Сводка по флагам "значение > 0":')
print(flags_df.sum().to_frame('rows_with_positive'))

# Полный крест по 4 ключевым полям
key_flags = [c for c in ['has_aur', 'has_retl_cnt', 'has_term_cnt', 'has_trx_sum'] if c in flags_df.columns]
if key_flags:
    print(f'\nКрест по флагам {key_flags}:')
    cross = flags_df.groupby(key_flags).size().reset_index(name='rows').sort_values('rows', ascending=False)
    display(cross)
else:
    print('\nНе нашлось ни одного из ожидаемых флагов для креста.')

# Сколько строк где ВСЁ ноль
zero_cols = [c for c in ['chod_num', 'finrez_num', 'term_cnt_num', 'retl_cnt_num',
                          'trx_sum_num', 'trx_cnt_num', 'aur_num', 'amortization_num']
             if c in excel_df.columns]
if zero_cols:
    mask_all_zero = excel_df[zero_cols].fillna(0).eq(0).all(axis=1)
    print(f'\nСтрок, где ВСЕ {len(zero_cols)} KPI равны 0/null: {int(mask_all_zero.sum()):,}')

    # Только AUR > 0, остальное 0
    if 'aur_num' in excel_df.columns:
        other_zero_cols = [c for c in zero_cols if c != 'aur_num']
        mask_only_aur = (
            (excel_df['aur_num'].fillna(0) > 0) &
            excel_df[other_zero_cols].fillna(0).eq(0).all(axis=1)
        )
        print(f'Строк где AUR > 0, остальное = 0: {int(mask_only_aur.sum()):,}')

## 7) Сэмплы строк разных типов

Показываем примеры:
- 10 строк где ВСЁ нули,
- 10 строк где только `AUR > 0`,
- 10 строк с самым большим оборотом (`trx_sum`),
- 10 строк с самым отрицательным `Фин.Рез`.

In [ ]:
display_cols = []
for label in ['inn', 'agr_id', 'contract_number']:
    if resolved.get(label):
        display_cols.append(resolved[label])
for label in ['retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum', 'aur', 'amortization',
              'commission_total', 'commission_from_ops', 'commission_monthly',
              'int_component', 'chod', 'finrez']:
    src = resolved.get(label)
    if src and src in excel_df.columns:
        display_cols.append(src)

display_cols = list(dict.fromkeys(display_cols))

if zero_cols:
    print('Сэмпл: ВСЕ нули')
    sample_zero = excel_df.loc[mask_all_zero, display_cols].head(10)
    display(sample_zero)

if zero_cols and 'aur_num' in excel_df.columns:
    print('\nСэмпл: AUR > 0, остальное 0')
    sample_aur_only = excel_df.loc[mask_only_aur, display_cols].head(10)
    display(sample_aur_only)

if 'trx_sum_num' in excel_df.columns:
    print('\nСэмпл: ТОП-10 по обороту')
    top_trx = excel_df.sort_values('trx_sum_num', ascending=False)[display_cols].head(10)
    display(top_trx)

if 'finrez_num' in excel_df.columns:
    print('\nСэмпл: ТОП-10 самых отрицательных Фин.Рез')
    bottom_finrez = excel_df.sort_values('finrez_num', ascending=True)[display_cols].head(10)
    display(bottom_finrez)

## 8) Финальная сводка

Что мы должны получить из этой тетрадки, чтобы понять структуру Excel:

| Вопрос | Где ответ |
|---|---|
| Сколько строк в файле? | блок 1 |
| Сколько уникальных ИНН и `agr_id`? | блок 4 |
| Есть ли в Excel колонка `agr_id` (или только `Номер договора`)? | блок 3 (resolved) |
| Какие KPI заполнены и какие пустые? | блок 5 |
| Какие сочетания нулей есть в Excel? | блок 6 (крест) |
| Как выглядят "пустые" строки (где почти всё 0)? | блок 7 (сэмплы) |
| Сколько agr_id у одного ИНН в Excel? | блок 4 (внизу) |

Если после запуска появятся непонятные моменты — пришли мне крест из блока 6 и сводку колонок из блока 2.